## Parte 4 

In [29]:
import numpy as np
from PIL import Image
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay 
from scipy.optimize import linear_sum_assignment
import os # Para leer todos los ficheros de la misma carpeta y no hacerlo a mano
import networkx as nx
from gensim.models import Word2Vec # Para hacer el node2vec


#------------------------------------------------------------------#

def load_and_preprocess_images(img_path1, img_path2, mat_path1, mat_path2):
    """
    Carga y preprocesa pares de imágenes y sus keypoints
    """
    # Definimos tamaño objeto
    obj_resize = (256,256)
    #Definimos las listas donde vamos a guardar los distintos elementos
    imgs = []
    kpts_list = []

    paths = [[img_path1, mat_path1],[img_path2, mat_path2]]

    for img_path, mat_path in paths:
        img = Image.open(img_path)
        kpts = np.array(sio.loadmat(mat_path)['pts_coord'])
        kpts[0] = kpts[0] * obj_resize[0] / img.size[0]
        kpts[1] = kpts[1] * obj_resize[1] / img.size[1]
        img = img.resize(obj_resize, resample=Image.BILINEAR)
        kpts_list.append(kpts)
        imgs.append(img)
    # Devolvemos los valores para poder usar la otra función
    return imgs[0], imgs[1], kpts_list[0], kpts_list[1]

#---------------------------------------------------------------#

def delaunay_triangulation(kpts):
    """
    Generate adjacency matrix based on Delaunay triangulation
    """
    # Transponemos para tener puntos como filas
    pts = kpts.T
    
    # Calculamos la triangulación de Delaunay
    tri = Delaunay(pts)
    
    # Inicializamos la matriz de adyacencia con ceros
    N = pts.shape[0]
    A = np.zeros((N, N), dtype=int)

    # Recorremos cada triángulo
    for triangle in tri.simplices:
        # triangle tiene 3 vértices: a, b, c
        a = triangle[0]
        b = triangle[1]
        c = triangle[2]

        # Conectamos a con b
        A[a, b] = 1
        A[b, a] = 1  # matriz simétrica

        # Conectamos b con c
        A[b, c] = 1
        A[c, b] = 1

        # Conectamos a comn c
        A[a, c] = 1
        A[c, a] = 1

    return A

#-------------------------------------------------------------------#

# Función para realizar una caminata aleatoria en el grafo
def node2vec_walk(G, start_node, walk_length):
    walk = [start_node]
    while len(walk) < walk_length:
        cur_node = walk[-1]
        neighbors = list(G.neighbors(cur_node))
        if len(neighbors) > 0:
            walk.append(np.random.choice(neighbors))  # Selecciona un vecino al azar
        else:
            break
    return walk

# Función para generar múltiples caminatas aleatorias desde cada nodo del grafo
def generate_walks(G, num_walks, walk_length):
    walks = []
    nodes = list(G.nodes())
    for _ in range(num_walks):
        np.random.shuffle(nodes)  # Mezcla los nodos para añadir aleatoriedad
        for node in nodes:
            walks.append(node2vec_walk(G, node, walk_length))  # Genera una caminata aleatoria desde el nodo
    return walks


def compute_node2vec_embeddings(G, dimensions=64, num_walks=10, walk_length=30):
    """
    Compute node2vec embeddings for the graph
    """
    walks = generate_walks(G, num_walks, walk_length)  # Genera todas las caminatas aleatorias
    walks = [list(map(str, walk)) for walk in walks]  # Convierte los nodos a cadenas para gensim
    # walks: Lista de caminatas aleatorias
    # vector_size=dimensions: Tamaño del vector de embeddings
    # window=5: Tamaño de la ventana de contexto, en nodos seria 5 nodos a la izquierda y 5 nodos a la derecha
    # workers=2: Número de hilos para entrenar el modelo
    model = Word2Vec(walks, vector_size=dimensions, window=5, workers=2)  # Entrena el modelo Word2Vec con las caminatas aleatorias
    embeddings = np.array([model.wv[str(node)] for node in G.nodes()])
    return embeddings

#------------------------------------------------------------------------------------#


def enhanced_spatial_matching(kpts1, kpts2, adj_matrix1, adj_matrix2):
    """
    Perform enhanced matching using spatial, hitting time, and node2vec features
 
    Args:
        kpts1: Array de keypoints del primer grafo (2xN)
        kpts2: Array de keypoints del segundo grafo (2xN)
        adj_matrix1: Matriz de adyacencia del primer grafo (NxN)
        adj_matrix2: Matriz de adyacencia del segundo grafo (NxN)
    
    Returns:
        matching: Matriz binaria donde matching[i,j]=1 indica correspondencia entre puntos
    Hints:
        - Primero deberás crear una matriz de costes basada en distancias euclidianas,
        cuyo tamaño será (n1, n2) donde n1 y n2 son el número de keypoints en cada grafo.
        - Seguidamente, deberás rellenar dicha matriz con las distancias euclidianas de los keypoints, las distancias/normas de node2vec 
        - A continuación, deberás aplicar el algoritmo húngaro usando la función linear_sum_assignment, que recibe la matriz de costes y devuelve los índices de los puntos emparejados.
        - Finalmente, deberás crear una matriz de matching a partir de los índices obtenidos.

    """
    # Create graphs
    n1, n2 = kpts1.shape[1], kpts2.shape[1]
    G1 = nx.from_numpy_array(adj_matrix1)
    G2 = nx.from_numpy_array(adj_matrix2)
    # print("Computing node2vec embeddings...")
    node2vec_emb1 = compute_node2vec_embeddings(G1)
    node2vec_emb2 = compute_node2vec_embeddings(G2)
    
    # Create cost matrix combining different features
    cost_matrix = np.zeros((n1, n2))
    
    for i in range(n1):
        for j in range(n2):
            # Spatial distance
            spatial_dist = np.sqrt(np.sum((kpts1[:,i] - kpts2[:,j])**2))
            
            # Node2vec similarity
            node2vec_dist = np.sqrt(np.sum((node2vec_emb1[i] - node2vec_emb2[j])**2))
            
            # Combine distances with weights
            cost_matrix[i,j] = (
                0.6 * spatial_dist +
                0.4 * node2vec_dist
            )
            
    row_index,col_index = linear_sum_assignment(cost_matrix)
    # Pasamos de un array de 2XE a un array de NxN
    matching = np.zeros((n1,n2))
    matching[row_index,col_index] = 1
    
    return matching


#--------------------------------------------------------------------------------#


def visualize_matching_full(img1, img2, kpts1, kpts2, adj_matrix1, adj_matrix2, matching):
    """
    Visualiza el matching completo entre dos grafos
    """
    
    # Graficar una imagen al lado de otra
    plt.figure(figsize=(15,8))
    # SI ya estan redimensionadas no haria falta (el max), simplemente se haria ancho = img1.shape
    ancho = max (img1.size[0],img2.size[0])
    alto = max (img1.size[1],img2.size[1])
    composicion = Image.new('RGB', (ancho*2, alto))
    composicion.paste(img1, (0,0))
    composicion.paste(img2, (ancho,0))
    plt.imshow(composicion)
    
    
    # Graficar las coordenadas (keypoints)
    plt.scatter(kpts1[0], kpts1[1], c='w', edgecolors='k', s=100)
    plt.scatter(kpts2[0] + ancho, kpts2[1], c='w', edgecolors='k', s=100)


    # Aristas imagen 1
    for i in range(adj_matrix1.shape[0]):
        for j in range(adj_matrix1.shape[1]):
            if adj_matrix1[i,j] == 1:
                plt.plot([kpts1[0,i], kpts1[0,j]],
                         [kpts1[1,i], kpts1[1,j]],
                         'y-', linewidth=0.5)

    # Aristas  imagen 2
    for i in range(adj_matrix2.shape[0]):
        for j in range(adj_matrix2.shape[1]):
            if adj_matrix2[i,j] == 1:
                plt.plot([kpts2[0,i] + ancho, kpts2[0,j] + ancho],
                         [kpts2[1,i], kpts2[1,j]],
                         'y-', linewidth=0.5)

    # Graficamoslas arsitas del matching
    for i in range(matching.shape[0]):
        for j in range(matching.shape[1]):
            if matching[i,j] > 0:
                color = 'g' if i==j else 'r'
                plt.plot([kpts1[0,i], kpts2[0,j] + ancho],
                         [kpts1[1,i], kpts2[1,j]],
                         color = color, linewidth=2)

## Ejercicio 2

In [ ]:
def mostar_pares(img_path1, mat_path1, img_path2, mat_path2, tipo):
    """
    Función que muestra imagenes
    """
    for i in range(2):
        img1, img2, kpt1, kpt2 = load_and_preprocess_images(img_path1[i], img_path2[i], mat_path1[i], mat_path2[i])
        
        # Aplicamos Delaunay
        if tipo == "Delaunay":
            matriz = delaunay_triangulation(kpt1)
            matriz2 = delaunay_triangulation(kpt2)
            # Hacemos el matching
            matching = enhanced_spatial_matching(kpt1, kpt2, matriz, matriz2)
            visualize_matching_full(img1, img2, kpt1, kpt2, matriz, matriz2, matching)
            plt.title(f'{tipo}')
        elif tipo == "k-NN":
            k = 5
            matriz = knn_graph(kpt1, k)
            matriz = knn_graph(kpt1, k)

             # Hacemos el matching
            matching = enhanced_spatial_matching(kpt1, kpt2, matriz, matriz2)
            visualize_matching_full(img1, img2, kpt1, kpt2, matriz, matriz2, matching)
            plt.title(f'{tipo}')

        else:
            return f"Tipo no disponible"

In [ ]:
def knn_graph(kpts, k):
    """Constructs a k-NN graph from keypoints."""
    # Implement k-NN graph construction
    pts = kpts.T
    lista_final = []

    # Calculamos las distancias euclidianas
    for i in range(len(pts)):
        punto = pts[i]
        for j in range(i+1, len(pts)):
            punto2 = pts[j]
            # distancia euclidiana correcta
            dist = math.sqrt((punto2[0] - punto[0])**2 + (punto2[1] - punto[1])**2)
            lista_final.append((i,j,dist))

    # Ya tenemos las distancias que hay entre cada punto clave
    edges = set()

    # Para cada punto, buscamos sus k vecinos más cercanos
    for i in range(len(pts)):

        distancias_i = []

        # Buscamos todas las distancias donde participe el punto i
        for (a, b, d) in lista_final:
            if a == i:
                distancias_i.append((d, a, b))
            elif b == i:
                distancias_i.append((d, b, a))

        # Ordenamos por distancia (de menor a mayor)
        distancias_i.sort()

        # Nos quedamos con los k más cercanos
        for t in range(min(k, len(distancias_i))):
            _, u, v = distancias_i[t]
            edges.add((u, v))
            
    return list(edges)

### CAR

In [ ]:
img_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Car/Cars_000a.png', './data/WillowObject/WILLOW-ObjectClass/Car/Cars_000a.png']
mat_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Car/Cars_000a.mat','./data/WillowObject/WILLOW-ObjectClass/Car/Cars_000a.mat']

img_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Car/Cars_008b.png','./data/WillowObject/WILLOW-ObjectClass/Car/Cars_008b.png']
mat_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Car/Cars_008b.mat','./data/WillowObject/WILLOW-ObjectClass/Car/Cars_008b.mat']

mostar_pares(img_path1, mat_path1, img_path2, mat_path2, "Delaunay")

### DUCK

In [ ]:
img_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Duck/060_0081.png', './data/WillowObject/WILLOW-ObjectClass/Duck/060_0081.png']
mat_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Duck/060_0081.mat','./data/WillowObject/WILLOW-ObjectClass/Duck/060_0081.mat']

img_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Duck/060_0066.png','./data/WillowObject/WILLOW-ObjectClass/Duck/060_0066.png']
mat_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Duck/060_0066.mat','./data/WillowObject/WILLOW-ObjectClass/Duck/060_0066.mat']

mostar_pares(img_path1, mat_path1, img_path2, mat_path2, "Delaunay")



### FACE

In [ ]:
img_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Face/image_0084.png', './data/WillowObject/WILLOW-ObjectClass/Face/image_0084.png']
mat_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Face/image_0084.mat','./data/WillowObject/WILLOW-ObjectClass/Face/image_0084.mat']

img_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Face/image_0092.png','./data/WillowObject/WILLOW-ObjectClass/Face/image_0092.png']
mat_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Face/image_0092.mat','./data/WillowObject/WILLOW-ObjectClass/Face/image_0092.mat']

mostar_pares(img_path1, mat_path1, img_path2, mat_path2, "Delaunay")

### MOTORBIKE

In [ ]:
img_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_001a.png', './data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_001a.png']
mat_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_001a.mat','./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_001a.mat']

img_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_005b.png','./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_005b.png']
mat_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_005b.mat','./data/WillowObject/WILLOW-ObjectClass/Motorbike/Motorbikes_005b.mat']

mostar_pares(img_path1, mat_path1, img_path2, mat_path2, "Delaunay")


### WINEBOTTLE

In [ ]:
img_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0061.png', './data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0061.png']
mat_path1 = ['./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0061.mat','./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0061.mat']

img_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0041.png','./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0041.png']
mat_path2 = ['./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0041.mat','./data/WillowObject/WILLOW-ObjectClass/Winebottle/246_0041.mat']

mostar_pares(img_path1, mat_path1, img_path2, mat_path2, "Delaunay")


## Ejercicio 3

### 1. Calcular la precisión para cada categoría

In [ ]:
def accuracy(matching):
    """
    Calculamos la precisión del matching entre dos imágenes

    Parámetros
    ----------
    - matching: Matching (necesario para la evaluación de la precisión)

    Fórmula
    -------
    - accuracy = nº puntos emparejados correctamente / nº total de puntos clave

    Funcionamiento
    --------------
    - Para saber si dos puntos claves se han emparejado correctamente, se cumple 
      la condición: matching[i,j] == 1 and i == j (cada vez que esto suceda, incrementamos
      en una unidad el contador de matches correctos)
    
    """

    matches_correctos = 0
    total_puntos = matching.shape[0]

    for i in range(matching.shape[0]):
        for j in range(matching.shape[1]):
            if matching[i,j] == 1 and i == j:
                matches_correctos += 1

    accuracy = matches_correctos / total_puntos

    return accuracy

def accuracy_categoria(category, tipo_matching, triangulation):
    """
    Calcula la precisión media de una categoría de imágenes.
    
    Devuelve:
    - Para Delaunay: media, desviación, número de pares
    - Para k-NN: lista de tuplas (media, desviación, n_pares) por cada k
    """

    path = './data/WillowObject/WILLOW-ObjectClass/' + category

    pngs, mats = [], []
    for img in os.listdir(path):
        if img.endswith('.png'):
            pngs.append(img)
        elif img.endswith('.mat'):
            mats.append(img)
    pngs = sorted(pngs)
    mats = sorted(mats)

    if triangulation == 'Delaunay':
        accuracies = []
        num_pares = 0
        for i in range(len(pngs)-1):
            for j in range(i+1, len(pngs)):
                img_path1 = path + '/' + pngs[i]
                mat_path1 = path + '/' + mats[i]
                img_path2 = path + '/' + pngs[j]
                mat_path2 = path + '/' + mats[j]

                img1, img2, kpt1, kpt2 = load_and_preprocess_images(img_path1, img_path2, mat_path1, mat_path2)

                if tipo_matching == 'simple':
                    matching = simple_spatial_matching(kpt1, kpt2)
                elif tipo_matching == 'enhanced':
                    adj1 = delaunay_triangulation(kpt1)
                    adj2 = delaunay_triangulation(kpt2)
                    matching = enhanced_spatial_matching(kpt1, kpt2, adj1, adj2)
                else:
                    return "Tipos: 'simple' | 'enhanced'"

                accuracies.append(accuracy(matching))
                num_pares += 1

        media = np.mean(accuracies)
        std = np.std(accuracies)
        return media, std, num_pares

    elif triangulation == 'k-NN':
        resultados_k = []
        for k in (1,3,5,7):
            accuracies = []
            num_pares = 0
            for i in range(len(pngs)-1):
                for j in range(i+1, len(pngs)):
                    img_path1 = path + '/' + pngs[i]
                    mat_path1 = path + '/' + mats[i]
                    img_path2 = path + '/' + pngs[j]
                    mat_path2 = path + '/' + mats[j]

                    img1, img2, kpt1, kpt2 = load_and_preprocess_images(img_path1, img_path2, mat_path1, mat_path2)

                    if tipo_matching == 'simple':
                        matching = simple_spatial_matching(kpt1, kpt2)
                    elif tipo_matching == 'enhanced':
                        adj1 = knn_graph(kpt1, k)
                        adj2 = knn_graph(kpt2, k)
                        matching = enhanced_spatial_matching(kpt1, kpt2, adj1, adj2)
                    else:
                        return "Tipos: 'simple' | 'enhanced'"

                    accuracies.append(accuracy(matching))
                    num_pares += 1

            resultados_k.append((np.mean(accuracies), np.std(accuracies), num_pares))

        return resultados_k

    else:
        return "Tipos: 'Delaunay' | 'k-NN'"

### Experimentos Obligatorios

### 1. Análisis baseline

In [32]:
def simple_spatial_matching(kpts1, kpts2):
    """
    Realiza matching entre puntos usando distancia euclidiana y algoritmo húngaro optimizado
    
    Args:
        kpts1: Array de keypoints del primer grafo (2xN)
        kpts2: Array de keypoints del segundo grafo (2xN)
    
    Returns:
        matching: Matriz binaria donde matching[i,j]=1 indica correspondencia entre puntos
    Hints:
        - Primero deberás crear una matriz de costes basada en distancias euclidianas,
        cuyo tamaño será (n1, n2) donde n1 y n2 son el número de keypoints en cada grafo.
        - Seguidamente, deberás rellenar dicha matriz con las distancias euclidianas entre
        los keypoints de ambos grafos.
        - A continuación, deberás aplicar el algoritmo húngaro usando la función linear_sum_assignment, que recibe la matriz de costes y devuelve los índices de los puntos emparejados.
        - Finalmente, deberás crear una matriz de matching a partir de los índices obtenidos.

    """
    
    n1, n2 = kpts1.shape[1], kpts2.shape[1]
    cost_matrix= np.zeros((n1,n2))
    for i in range(n1):
        for j in range(n2):
            cost_matrix[i,j] = np.sqrt(np.sum((kpts1[:,i] - kpts2[:,j])**2))
    
    row_index,col_index = linear_sum_assignment(cost_matrix)
    # Pasamos de un array de 2XE a un array de NxN
    matching = np.zeros((n1,n2))
    matching[row_index,col_index] = 1
    return matching

#### Análisis datos de correspondecfia mejorar (con Delaunay)

In [ ]:
# Calculamos el accuracy de las 5 categorías usando el 
# simple_spatial_matching y la triangulación de Delaunay

categories = ['Car', 'Duck', 'Face', 'Motorbike', 'Winebottle']
for category in categories:
    acc, std, n = accuracy_categoria(category, tipo_matching='simple', triangulation = 'Delaunay')
    print(f'INFO DE LA CATEGORÍA\n---------------------\n\
    Categoría: {category}\n\
    Precisión: {acc:.2f}\n\
    Desviación: {std:.2f}\n\
    Pares evaluados: {n}\n')

INFO DE LA CATEGORÍA
---------------------
    Categoría: Car
    Precisión: 0.72
    Desviación: 0.32
    Pares evaluados: 780
INFO DE LA CATEGORÍA
---------------------
    Categoría: Duck
    Precisión: 0.77
    Desviación: 0.30
    Pares evaluados: 1225
INFO DE LA CATEGORÍA
---------------------
    Categoría: Face
    Precisión: 0.87
    Desviación: 0.18
    Pares evaluados: 5886
INFO DE LA CATEGORÍA
---------------------
    Categoría: Motorbike
    Precisión: 0.88
    Desviación: 0.19
    Pares evaluados: 780
INFO DE LA CATEGORÍA
---------------------
    Categoría: Winebottle
    Precisión: 0.92
    Desviación: 0.17
    Pares evaluados: 2145


#### Análisis datos de correspondecfia mejorar (con Delaunay)

In [ ]:
# Calculamos el accuracy de las 5 categorías uando el 
# enhanced_spatial_matching y la triangulación de Delaunay

categories = ['Car', 'Duck', 'Face', 'Motorbike', 'Winebottle']
for category in categories:
    acc, std, n = accuracy_categoria(category, tipo_matching='enhanced', triangulation ='Delaunay')
    print(f'INFO DE LA CATEGORÍA\n-------------\n\
        Categoría: {category}\n\
        Precisión: {acc:.2f}\n\
        Desviación: {std:.2f}\n\
        Pares evaluados: {n}\n')

INFO DE LA CATEGORÍA
-------------
        Categoría: Car
        Precisión: 0.73
        Desviación: 0.32
        Pares evaluados: 780

INFO DE LA CATEGORÍA
-------------
        Categoría: Duck
        Precisión: 0.77
        Desviación: 0.30
        Pares evaluados: 1225



### 3. Análisis de K-vecinos más cercanos

In [ ]:
category = 'Duck'
k = [1,3,5,7]

resultados_k = accuracy_categoria('Duck', tipo_matching='enhanced', triangulation = 'k-NN')

for i in range(len(resultados_k)):
    acc, std, n = resultados_k[i]
    k_usado = k[i]

    print(f'Datos para k={k_usado}\n----------------------\n\
        Categoría: {category}\n\
        Precisión: {acc:.2f}\n\
        Desviación: {std:.2f}\n\
        Pares evaluados: {n}\n')

## Ejercicio 3. Evaluar los resultados de la correspondencia e informar

In [ ]:
def accuracy(matching):
    """
    Calculamos la precisión del matching entre dos imágenes

    Parámetros
    ----------
    - matching: Matching (necesario para la evaluación de la precisión)

    Fórmula
    -------
    - accuracy = nº puntos emparejados correctamente / nº total de puntos clave

    Funcionamiento
    --------------
    - Para saber si dos puntos claves se han emparejado correctamente, se cumple 
      la condición: matching[i,j] == 1 and i == j (cada vez que esto suceda, incrementamos
      en una unidad el contador de matches correctos)
    
    """

    matches_correctos = 0
    total_puntos = matching.shape[0]

    for i in range(matching.shape[0]):
        for j in range(matching.shape[1]):
            if matching[i,j] == 1 and i == j:
                matches_correctos += 1

    accuracy = matches_correctos / total_puntos

    return accuracy

#--------------------------------------------------------------------------------#
def accuracy_categoria(category):
    """
    Función que recorre las imágenes de una categoría, va realizando matching entre 
    ellas y va calculando preciones. Una vez tiene una lista de precisiones, se 
    calcula la media para saber el accuracy de la categoría
    """

    path = './data/WillowObject/WILLOW-ObjectClass/' + category

    # Listas para guardar las imágenes y archivos .mat
    pngs = []
    mats = []

    for img in os.listdir(path):
        if img.endswith('.png'):
            pngs.append(img)
        elif img.endswith('.mat'):
            mats.append(img)
    
    pngs = sorted(pngs)
    mats = sorted(mats)

    accuracies = []
    num_pares = 0

    # Vamos a recorrer la categoría
    for i in range(len(pngs)-1):
        for j in range(i + 1, len(pngs)):
            img_path1 = path + '/' + pngs[i]
            mat_path1 = path + '/' + mats[i]

            img_path2 = path + '/' + pngs[j]
            mat_path2 = path + '/' + mats[j]

            # Una vez que tenemos las rutas de las imagenes, aplicamos el procedimiento anterior
            img1, img2, kpt1, kpt2 = load_and_preprocess_images(img_path1, img_path2, mat_path1, mat_path2)

            # Hacemos el matching
            matching = enhanced_spatial_matching(kpt1, kpt2)

            # Calculamos el accuracy
            acc = accuracy(matching)
            accuracies.append(acc)
            num_pares += 1

    # Calculamos la media
    x = len(accuracies)
    total = sum(accuracies)
    media = total/x

    # Calculamos la desviación típica
    desviacion = np.std(accuracies)

    return media, desviacion, num_pares

Función para calcular 